# Exercise E — Turn Neo4j RAG into *Real* GraphRAG
The graph-DB app stores isolated `:Chunk` nodes with no relationships — so it's RAG on a graph database, not GraphRAG. Here you add **entities** and **relationships**, then retrieve by vector AND traverse links (multi-hop). This is the real thing.

**Offline scaffold:** builds the entity graph in Python + prints the Cypher you'd run in Neo4j. The concepts transfer directly.

In [ ]:
# Offline mock so this scaffold runs with NO API key / NO network.
# For real practice, replace `embed()` with your real embedder (InHouseEmbeddings,
# SentenceTransformer, etc.) and `llm()` with a real model call.
import numpy as np, re
_STOP=set("the a an to of and or is are be for in on at by with from as that this it its".split())
def _tok(t): return [w for w in re.findall(r"[a-z0-9]+",t.lower()) if w not in _STOP and len(w)>2]
def embed(texts):
    if isinstance(texts,str): texts=[texts]
    out=[]
    for t in texts:
        v=np.zeros(256)
        for w in _tok(t): v[abs(hash(w))%256]+=1
        n=np.linalg.norm(v); out.append(v/n if n else v)
    return np.array(out)
def cos(a,b): return float(a@b)

# A small corpus standing in for chunks of ERP-2008-chapter4.pdf (health-care economics).
CORPUS = [
 ("Demand for health care is derived from the value of improved health, not the procedures themselves.","demand"),
 ("Health can be defined by longevity (length of life) and quality of life.","demand"),
 ("National health spending reached over 7000 dollars per capita and about 16 percent of GDP.","spending"),
 ("Medical technology accounts for about half of long-term health spending growth.","spending"),
 ("Medicare, enacted in 1965, covers people aged 65 and older; Part D is the drug benefit.","medicare"),
 ("Medicaid, established in 1965, is a program for low-income individuals, administered by states.","medicaid"),
 ("Moral hazard is the tendency to overuse care when insurance covers most of the cost.","moral_hazard"),
 ("Adverse selection is when insurance is most attractive to those most likely to need it.","insurance"),
 ("Health Savings Accounts use pre-tax dollars with high-deductible plans to reduce routine-care reliance.","hsa"),
 ("The proposed standard deduction for health insurance would be a flat 15000 dollars per family.","tax"),
]
texts=[c[0] for c in CORPUS]; sections=[c[1] for c in CORPUS]
print("Mock corpus ready:", len(texts), "chunks.")

In [ ]:
# 1. Extract entities from each chunk (here: keyword match; real version uses an LLM/NER)
ENTITIES = ["Medicare","Medicaid","HSA","moral hazard","adverse selection","health","GDP"]
def entities_in(text):
    return [e for e in ENTITIES if e.lower() in text.lower()]

chunk_entities = {i: entities_in(t) for i,t in enumerate(texts)}
for i,es in chunk_entities.items():
    if es: print(f"chunk {i}: {es}")

In [ ]:
# 2. The Cypher you'd run in Neo4j to build the graph (print, don't execute here)
print("// For each chunk, link it to the entities it mentions:")
print("""
UNWIND $rows AS row
MERGE (c:Chunk {id: row.chunk_id})
WITH c, row
UNWIND row.entities AS ename
  MERGE (e:Entity {name: ename})
  MERGE (c)-[:MENTIONS]->(e);
""")
print("// Then relate entities that co-occur in the same chunk:")
print("""
MATCH (c:Chunk)-[:MENTIONS]->(e1:Entity)
MATCH (c)-[:MENTIONS]->(e2:Entity)
WHERE id(e1) < id(e2)
MERGE (e1)-[:RELATED_TO]->(e2);
""")

In [ ]:
# 3. Multi-hop retrieval, simulated: vector-match a chunk, then pull in chunks that share
#    an entity with it (the "graph" step plain vector RAG can't do).
qv = embed("how does insurance cause overuse of care?")[0]
seed = max(range(len(texts)), key=lambda i: cos(qv, embed(texts[i])[0]))
print("Vector seed chunk:", seed, "->", texts[seed][:70])
seed_ents = set(chunk_entities[seed])
print("Its entities:", seed_ents)

# hop: any chunk sharing an entity with the seed
neighbors = [i for i in range(len(texts)) if i!=seed and set(chunk_entities[i]) & seed_ents]
print("\nGraph-expanded context (shares an entity):")
for i in neighbors: print(f"  [{i}] {texts[i][:70]}  (shared: {set(chunk_entities[i])&seed_ents})")

### Observe & decide
- Plain vector RAG returns only the seed chunk. Graph expansion pulls in *related* chunks via shared entities — that's multi-hop retrieval, the thing 'graph' adds.
**Your turn:** implement steps 1–2 against the real Neo4j app (add entity extraction to `pdf_processor.py`, the Cypher to `vector_store.py`), then modify `search_similar` to traverse `MENTIONS`/`RELATED_TO`. This is the capstone — ties to Specialist Track multi-hop.